<a href="https://colab.research.google.com/github/JonasFanZ/114-2-Programing-Language/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://docs.google.com/spreadsheets/d/1kdaLflcZT60JRD9QKvELQvaY6SRj9M014uLVetrZHYE/edit?usp=sharing

In [1]:
import gradio as gr
import gspread
from google.colab import auth, userdata
from google.auth import default
import pandas as pd
import plotly.graph_objects as go
import google.generativeai as genai
import re

# ==========================================
# 0. 全域變數設定
# ==========================================
# ⚠️ 請替換成你的 Google 試算表連結
SHEET_URL = "https://docs.google.com/spreadsheets/d/1kdaLflcZT60JRD9QKvELQvaY6SRj9M014uLVetrZHYE/edit?usp=sharing"
MAX_STUDENTS = 60  # 專注模式最多支援的單班人數

# ==========================================
# 1. 認證與初始化
# ==========================================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

try:
    gemini_api_key = userdata.get('gemini')
    genai.configure(api_key=gemini_api_key)
    gemini_model = genai.GenerativeModel('gemini-2.5-flash')
except Exception as e:
    print("⚠️ 警告：無法讀取 Gemini API Key。")

# ==========================================
# 2. 核心功能函數
# ==========================================
def get_spreadsheet():
    return gc.open_by_url(SHEET_URL)

def get_class_list():
    try:
        wb = get_spreadsheet()
        return [sheet.title for sheet in wb.worksheets() if sheet.title not in ["工作表1", "Sheet1"]]
    except:
        return []

def update_all_dropdowns():
    classes = get_class_list()
    return (
        gr.update(choices=classes),
        gr.update(choices=classes),
        gr.update(choices=classes),
        gr.update(choices=classes)
    )

def create_class(class_name, subjects_str):
    if not class_name or not subjects_str:
        gr.Warning("班級名稱與科目不能為空！")
        return update_all_dropdowns()
    try:
        wb = get_spreadsheet()
        subjects = [s.strip() for s in subjects_str.split(',')]
        worksheet = wb.add_worksheet(title=class_name, rows="100", cols=str(len(subjects) + 2))
        worksheet.append_row(['學號', '姓名'] + subjects)
        gr.Info(f"✅ 成功建立班級「{class_name}」")
    except Exception as e:
        gr.Warning(f"❌ 建立失敗：{str(e)}")
    return update_all_dropdowns()

def add_students(class_name, students_text):
    if not class_name or not students_text:
        gr.Warning("請選擇班級並輸入名單！")
        return
    try:
        worksheet = get_spreadsheet().worksheet(class_name)
        lines = students_text.strip().split('\n')
        new_rows = []
        for line in lines:
            parts = re.split(r'[,\s]+', line.strip())
            if len(parts) >= 2:
                new_rows.append([str(parts[0]), str(parts[1])] + [""] * (worksheet.col_count - 2))
        if new_rows:
            worksheet.append_rows(new_rows)
            gr.Info(f"✅ 成功匯入 {len(new_rows)} 位學生！")
        else:
            gr.Warning("❌ 格式錯誤。")
    except Exception as e:
        gr.Warning(f"❌ 匯入失敗：{str(e)}")

# --- 專注模式 (極速單科打分) 相關函數 ---
def update_speed_subjects(class_name):
    if not class_name: return gr.update(choices=[], value=None)
    try:
        headers = get_spreadsheet().worksheet(class_name).row_values(1)
        return gr.update(choices=headers[2:], value=None)
    except:
        return gr.update(choices=[], value=None)

def load_speed_grading(class_name, subject_name):
    empty_returns = [gr.update(visible=False)]*MAX_STUDENTS + [gr.update(value="")]*MAX_STUDENTS + [gr.update(value=None)]*MAX_STUDENTS + [[], "等待操作..."]
    if not class_name or not subject_name: return empty_returns

    try:
        df = pd.DataFrame(get_spreadsheet().worksheet(class_name).get_all_records())
        if subject_name not in df.columns: return empty_returns

        student_ids = df['學號'].astype(str).tolist()
        student_names = df['姓名'].astype(str).tolist()
        current_scores = df[subject_name].tolist()

        rows_update, names_update, scores_update = [], [], []

        for i in range(MAX_STUDENTS):
            if i < len(student_ids):
                rows_update.append(gr.update(visible=True))
                names_update.append(gr.update(value=f"### 👤 {student_ids[i]} - {student_names[i]}"))
                val = float(current_scores[i]) if current_scores[i] != "" else None
                scores_update.append(gr.update(value=val))
            else:
                rows_update.append(gr.update(visible=False))
                names_update.append(gr.update(value=""))
                scores_update.append(gr.update(value=None))

        return rows_update + names_update + scores_update + [student_ids, f"✅ 已載入 {class_name} 的 {subject_name} 成績。請點擊第一個欄位並用 Enter 快速往下！"]
    except Exception as e:
        gr.Warning(f"讀取失敗：{str(e)}")
        return empty_returns

def save_speed_grades(class_name, subject_name, student_ids, *scores):
    if not class_name or not subject_name or not student_ids:
        gr.Warning("請先載入科目資料")
        return "❌ 儲存失敗"
    try:
        worksheet = get_spreadsheet().worksheet(class_name)
        headers = worksheet.row_values(1)
        col_index = headers.index(subject_name) + 1

        a1_notation = gspread.utils.rowcol_to_a1(1, col_index)
        col_letter = ''.join([i for i in a1_notation if not i.isdigit()])
        range_name = f"{col_letter}2:{col_letter}{len(student_ids)+1}"

        values = [[str(scores[i]) if scores[i] is not None else ""] for i in range(len(student_ids))]
        worksheet.update(range_name=range_name, values=values)
        gr.Info(f"💾 {subject_name} 儲存成功！")
        return f"✅ {subject_name} 儲存成功！"
    except Exception as e:
        return f"❌ 儲存失敗：{str(e)}"

# --- 分布圖相關函數 (修正 Y 軸整數顯示) ---
def update_distribution_subjects(class_name):
    if not class_name: return gr.update(choices=[])
    try:
        headers = get_spreadsheet().worksheet(class_name).row_values(1)
        return gr.update(choices=["平均分數"] + headers[2:], value="平均分數")
    except:
        return gr.update(choices=[])

def generate_distribution_chart(class_name, target_col):
    if not class_name or not target_col: return None
    try:
        df = pd.DataFrame(get_spreadsheet().worksheet(class_name).get_all_records())
        df_numeric = df.drop(columns=['學號', '姓名']).apply(pd.to_numeric, errors='coerce')

        if target_col == "平均分數":
            data = df_numeric.mean(axis=1).fillna(0)
            title = f"{class_name} - 全班平均分數分布"
            color = "#8b5cf6"
        else:
            if target_col not in df_numeric.columns: return None
            data = df_numeric[target_col].fillna(0)
            title = f"{class_name} - {target_col} 成績分布"
            color = "#3b82f6"

        fig = go.Figure(data=[go.Histogram(x=data, xbins=dict(start=0, end=100, size=10), marker_color=color, opacity=0.8)])

        # 動態決定 Y 軸刻度：如果全班人數較少，強制間距為 1，否則由系統自動分配整數刻度
        student_count = len(data)
        dtick_val = 1 if student_count <= 20 else None

        fig.update_layout(
            title=title,
            xaxis_title="分數區間",
            # 修正此處：強制格式為整數 "d"，並加入 dtick 避免小數點
            yaxis=dict(title="學生人數", tickformat="d", dtick=dtick_val),
            bargap=0.1,
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)'
        )
        return fig
    except:
        return None

# --- 個體分析相關函數 ---
def get_students_for_analysis(class_name):
    if not class_name: return gr.update(choices=[])
    try:
        df = pd.DataFrame(get_spreadsheet().worksheet(class_name).get_all_records())
        choices = [f"{row['學號']} - {row['姓名']}" for _, row in df.iterrows()]
        return gr.update(choices=choices, value=None)
    except:
        return gr.update(choices=[])

def generate_analysis(class_name, student_selected):
    if not class_name or not student_selected: return None, "", ""
    try:
        worksheet = get_spreadsheet().worksheet(class_name)
        df = pd.DataFrame(worksheet.get_all_records())

        student_id = str(student_selected.split(" - ")[0])
        student_name = student_selected.split(" - ")[1]

        df_numeric = df.drop(columns=['學號', '姓名']).apply(pd.to_numeric, errors='coerce')
        class_avg = df_numeric.mean().fillna(0).tolist()
        subjects = df_numeric.columns.tolist()

        student_row = df[df['學號'].astype(str) == student_id]
        scores = student_row.drop(columns=['學號', '姓名']).apply(pd.to_numeric, errors='coerce').iloc[0].fillna(0).tolist()

        avg_score = sum(scores) / len(scores) if len(scores) > 0 else 0
        if avg_score >= 60:
            status_html = f"<div style='color: #166534; background-color: #dcfce7; padding: 12px; border-radius: 8px; font-weight: bold; text-align: center; font-size: 18px;'>🟢 學習狀態良好 (平均: {avg_score:.1f})</div>"
        else:
            status_html = f"<div style='color: #991b1b; background-color: #fee2e2; padding: 12px; border-radius: 8px; font-weight: bold; text-align: center; font-size: 18px;'>🔴 需加強關注 (平均: {avg_score:.1f})</div>"

        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(r=class_avg + [class_avg[0]], theta=subjects + [subjects[0]], fill='toself', name='全班平均', fillcolor='rgba(100, 116, 139, 0.2)', line=dict(color='#64748b')))
        fig.add_trace(go.Scatterpolar(r=scores + [scores[0]], theta=subjects + [subjects[0]], fill='toself', name=student_name, fillcolor='rgba(59, 130, 246, 0.4)', line=dict(color='#3b82f6', width=2)))
        fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])), showlegend=True, margin=dict(l=30, r=30, t=30, b=30), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')

        prompt = f"""以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n學生姓名：{student_name}\n各科成績：{dict(zip(subjects, scores))}\n全班平均：{dict(zip(subjects, [round(avg, 1) for avg in class_avg]))}"""
        response = gemini_model.generate_content(prompt)

        return fig, response.text, status_html
    except Exception as e:
        return None, f"⚠️ 分析失敗：{str(e)}", ""

# ==========================================
# 3. UI 介面設計與 JS 事件注入
# ==========================================
js_injection = """
function() {
    window.addEventListener('keydown', function(e) {
        if (e.key === 'Enter') {
            let active = document.activeElement;
            if (active && active.tagName === 'INPUT') {
                let inputs = Array.from(document.querySelectorAll('input:not([disabled])'));
                inputs = inputs.filter(el => el.offsetParent !== null);

                let idx = inputs.indexOf(active);
                if (idx > -1 && idx < inputs.length - 1) {
                    e.preventDefault();
                    inputs[idx + 1].focus();
                    inputs[idx + 1].select();
                }
            }
        }
    });
}
"""

custom_css = """
.glass-panel { background-color: #ffffff !important; border: 1px solid #e2e8f0 !important; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1) !important; border-radius: 12px !important; padding: 20px !important; }
.primary-btn { transition: filter 0.2s ease-in-out !important; }
.primary-btn:hover { filter: brightness(0.9) !important; }
.compact-row { align-items: center !important; border-bottom: 1px solid #f1f5f9; padding-bottom: 5px; margin-bottom: 5px; }
"""

custom_theme = gr.themes.Soft(
    primary_hue="blue", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Optima"), "system-ui", "sans-serif"],
).set(
    body_background_fill="#f8fafc", block_background_fill="#ffffff",
    block_border_width="1px", container_radius="12px", button_large_radius="8px"
)

with gr.Blocks(theme=custom_theme, css=custom_css) as demo:
    gr.Markdown("# 📊 智慧班級成績分析系統", elem_classes="glass-panel")

    current_student_ids = gr.State([])

    with gr.Tabs():
        # --- 分頁 1: 系統設定 ---
        with gr.TabItem("⚙️ 系統設定"):
            with gr.Accordion("建立班級與匯入名單", open=True, elem_classes="glass-panel"):
                with gr.Row():
                    with gr.Column():
                        class_name_input = gr.Textbox(label="班級名稱", placeholder="例如：101")
                        subjects_input = gr.Textbox(label="測驗科目 (請用逗號分隔)")
                        create_class_btn = gr.Button("➕ 建立班級", variant="primary", elem_classes="primary-btn")
                    with gr.Column():
                        target_class_dropdown = gr.Dropdown(label="選擇班級", choices=[])
                        students_input = gr.Textbox(label="學生名單 (學號, 姓名)", lines=4)
                        add_students_btn = gr.Button("📥 批次匯入名單", variant="primary", elem_classes="primary-btn")

        # --- 分頁 2: ⚡ 專注模式 ---
        with gr.TabItem("⚡ 極速成績登錄"):
            with gr.Column(elem_classes="glass-panel"):
                gr.Markdown("### 🎯 選擇單一科目，在第一個格子輸入後狂按 `Enter` 鍵即可光速盲打！")

                with gr.Row():
                    speed_class_dropdown = gr.Dropdown(label="選擇班級", choices=[], scale=2)
                    speed_subject_dropdown = gr.Dropdown(label="選擇科目", choices=[], scale=2)
                    speed_save_btn = gr.Button("💾 儲存本科成績", variant="primary", scale=1, elem_classes="primary-btn")

                speed_status = gr.Markdown("等待操作...")
                gr.Markdown("---")

                row_comps, name_comps, score_comps = [], [], []

                for i in range(MAX_STUDENTS):
                    with gr.Row(visible=False, elem_classes="compact-row") as r:
                        with gr.Column(scale=3, min_width=150):
                            n = gr.Markdown(f"姓名 {i}")
                        with gr.Column(scale=1, min_width=100):
                            s = gr.Number(label="分數", container=False)
                    row_comps.append(r)
                    name_comps.append(n)
                    score_comps.append(s)

        # --- 分頁 3: 班級整體分布 ---
        with gr.TabItem("📊 班級整體分布"):
            with gr.Column(elem_classes="glass-panel"):
                with gr.Row():
                    dist_class_dropdown = gr.Dropdown(label="選擇班級", choices=[], scale=2)
                    dist_subject_dropdown = gr.Dropdown(label="選擇查看目標", choices=[], scale=2)
                    generate_dist_btn = gr.Button("📈 產生分布圖", variant="primary", scale=1, elem_classes="primary-btn")

                distribution_plot = gr.Plot(label="成績分布直方圖")

        # --- 分頁 4: 個體分析 ---
        with gr.TabItem("🔍 個體深度分析"):
            with gr.Column(elem_classes="glass-panel"):
                with gr.Row():
                    analyze_class_dropdown = gr.Dropdown(label="選擇班級", choices=[])
                    analyze_student_dropdown = gr.Dropdown(label="選擇學生", choices=[])
                    analyze_btn = gr.Button("✨ 產生報告", variant="primary", elem_classes="primary-btn")

                student_status_html = gr.HTML("")
                with gr.Row():
                    with gr.Column(scale=6):
                        radar_plot = gr.Plot(label="學習能力雷達圖")
                    with gr.Column(scale=4):
                        ai_summary_output = gr.Textbox(label="Gemini 學習摘要", lines=18, interactive=False)

    # --- 啟動時綁定所有事件與 JS ---
    demo.load(fn=update_all_dropdowns, outputs=[target_class_dropdown, speed_class_dropdown, analyze_class_dropdown, dist_class_dropdown], js=js_injection)

    create_class_btn.click(fn=create_class, inputs=[class_name_input, subjects_input], outputs=[target_class_dropdown, speed_class_dropdown, analyze_class_dropdown, dist_class_dropdown])
    add_students_btn.click(fn=add_students, inputs=[target_class_dropdown, students_input], outputs=[])

    speed_class_dropdown.change(fn=update_speed_subjects, inputs=[speed_class_dropdown], outputs=[speed_subject_dropdown])

    speed_subject_dropdown.change(
        fn=load_speed_grading,
        inputs=[speed_class_dropdown, speed_subject_dropdown],
        outputs=row_comps + name_comps + score_comps + [current_student_ids, speed_status]
    )

    speed_save_btn.click(
        fn=save_speed_grades,
        inputs=[speed_class_dropdown, speed_subject_dropdown, current_student_ids] + score_comps,
        outputs=[speed_status]
    )

    dist_class_dropdown.change(fn=update_distribution_subjects, inputs=[dist_class_dropdown], outputs=[dist_subject_dropdown])
    generate_dist_btn.click(fn=generate_distribution_chart, inputs=[dist_class_dropdown, dist_subject_dropdown], outputs=[distribution_plot])

    analyze_class_dropdown.change(fn=get_students_for_analysis, inputs=[analyze_class_dropdown], outputs=[analyze_student_dropdown])
    analyze_btn.click(fn=generate_analysis, inputs=[analyze_class_dropdown, analyze_student_dropdown], outputs=[radar_plot, ai_summary_output, student_status_html])

# 忽略 DeprecationWarning，直接啟動
demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_10035/3712907512.py:273: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css) as demo:
/tmp/ipykernel_10035/3712907512.py:273: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6e3735034e97b3f188.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6e3735034e97b3f188.gradio.live
